In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
    
from waffles.data_classes.EasyWaveformCreator import EasyWaveformCreator
from waffles.data_classes.ChannelWsGrid import ChannelWsGrid
from waffles.np02_utils.load_utils import ch_read_template
from waffles.np02_utils.AutoMap import getEndpointChannelFromModule
from waffles.np02_utils.PlotUtils import plot_waveforms_wfset
from waffles.np02_data import templates as waffletemplates
from utils import save_templates 

In [ ]:
templates_name = 'templates_large_pulses'

templates = ch_read_template(templates_name)
templates_zero = ch_read_template(f'{templates_name}_no_led')

In [ ]:
foldertemplates = waffletemplates.__path__[0]

In [ ]:
original_amplitudes = {}
for mod in ['cathode', 'membrane']:
# for mod in ['pmt']:
    readmetemplate = foldertemplates + f'/{templates_name}/wfdetails_{mod}.info'
    with open(readmetemplate) as f:
        lines = f.readlines()
        lines = [ line.strip() for line in lines if line.strip()]
        if lines[0].startswith('Module'):
            lines = lines[1:] # remove header
    
        for line in lines:
            data = line.split()
            module = data[0]
            amplitude = data[-1]
            ep, ch = getEndpointChannelFromModule(module)
            original_amplitudes[(ep,ch)] = float(amplitude)
    


In [ ]:
templates_real_scale = { ep: { ch: values*original_amplitudes[(ep,ch)]/np.max(values) for ch, values in chv.items() } for ep, chv in templates.items() }
templates_subtracked = { ep: { ch: values - templates_zero[ep][ch] for ch, values in chv.items() } for ep, chv in templates_real_scale.items() }
peaks = { (ep, ch): np.max(v) for ep, chv in templates_subtracked.items() for ch, v in chv.items() }
templates_subtracked_norm = { ep: { ch: values*np.max(templates[ep][ch])/np.max(values) for ch, values in chv.items() } for ep, chv in templates_subtracked.items() }
dict_ep_ch = { ep: list(cht.keys()) for ep, cht in templates.items() }

In [ ]:
wffake = EasyWaveformCreator.create_WaveformSet_dictEndpointCh(dict_endpoint_ch=dict_ep_ch)

In [ ]:
fig, g = plot_waveforms_wfset(wfset=wffake, waveforms=templates, detectors=['C', 'M'], cols=4, shared_xaxes=True, shared_yaxes=True)
fig, g = plot_waveforms_wfset(wfset=wffake, waveforms=templates_subtracked_norm, detectors=['C', 'M'], cols=4, fig=fig)
fig.update_layout(template="plotly_white", width=1600, height=1600, showlegend=False)
fig.update_annotations( font=dict(size=14), align="center", )
fig.show()

In [ ]:
wfsetch = ChannelWsGrid.clusterize_waveform_set(wffake)

In [ ]:
wfsetch_cathode = {}
wfsetch_membrane = {}
wfsetch_pmt = {}
wfsetch_cathode[106] = wfsetch[106]
wfsetch_membrane[107] = wfsetch[107]
# wfsetch_pmt[110] = wfsetch[110]

In [ ]:
subtracted_template_name = f'{templates_name}_zero_subtract'
cuts_used = foldertemplates + f'/{templates_name}/cuts_used.yaml' 
save_templates(wfsetch_cathode, template_name=subtracted_template_name, cutyaml=cuts_used, detector=['C'], templates=templates_subtracked_norm, peaks=peaks, dry_run=False)
save_templates(wfsetch_membrane, template_name=subtracted_template_name, cutyaml=cuts_used, detector=['M'], templates=templates_subtracked_norm, peaks=peaks, dry_run=False)
# save_templates(wfsetch_pmt, template_name=subtracted_template_name, cutyaml=cuts_used, detector=['P'], templates=templates_subtracked_norm, peaks=peaks, dry_run=False)